In [2]:
from unstructured.partition.pdf import partition_pdf

c:\Users\Srinivasan\Documents\skills\10_days\Day10\aiquiz\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from unstructured.partition.pdf import partition_pdf
# import pytesseract
# tesseract =r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# pytesseract.pytesseract.tesseract_cmd = tesseract
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "C:/Users/Srinivasan/Documents/skills/10_days/Day10/aiquiz/files/osds.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: C:/Users/Srinivasan/Documents/skills/10_days/Day10/aiquiz/files/osds.pdf


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 4456.19it/s]


✅ Extracted 88 elements


In [5]:
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [6]:
elements[36].to_dict()

{'type': 'NarrativeText',
 'element_id': '2a689ea0c82981b4ff905bb1027e19ec',
 'text': 'A file system is the method an OS uses to organize and retrieve data on a storage device. It manages directories, metadata such as file size and timestamps, permissions, and physical block allocation. Common file systems include FAT32 and NTFS on Windows, ext4 on Linux, and APFS on macOS. Each has different strengths in terms of file size limits, journaling support, and performance.',
 'metadata': {'detection_class_prob': 0.9456956386566162,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(414.99344444444444),
     np.float64(806.634305555556)),
    (np.float64(414.99344444444444), np.float64(1167.02783203125)),
    (np.float64(2493.42431640625), np.float64(1167.02783203125)),
    (np.float64(2493.42431640625), np.float64(806.634305555556))),
   'system': 'PixelSpace',
   'layout_width': 2894,
   'layout_height': 4093},
  'last_modified': '2026-04-30T01:21:55',
  'filetype': 'applic

In [7]:
from unstructured.chunking.title import chunk_by_title
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=800, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=740, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=300 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 31 chunks


In [9]:
from langchain_openai import OpenAIEmbeddings
import os 
from dotenv import load_dotenv
load_dotenv()
api=os.getenv('OPEN_API_KEY')
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api,
    base_url="https://openrouter.ai/api/v1"
)

In [10]:
from langchain_chroma import Chroma
from langchain_core.documents import Document


def chunks_to_documents(chunks, filename: str):
    """Convert unstructured chunks into LangChain documents with minimal metadata."""
    documents = []

    for index, chunk in enumerate(chunks):
        chunk_data = chunk.to_dict()
        chunk_metadata = chunk_data.get("metadata") or {}

        documents.append(
            Document(
                page_content=chunk_data.get("text", ""),
                metadata={
                    "filename": filename,
                    "page_number": chunk_metadata.get("page_number"),
                    "index_number": index,
                },
            )
        )

    return documents


filename = "./files/osds.pdf"
documents = chunks_to_documents(chunks, filename)

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="osds_chunks",
    persist_directory="./chroma_db",
)

print(f"✅ Embedded and stored {len(documents)} chunks in Chroma")
print("Collection: osds_chunks")
print("Persist directory: ./chroma_db")

✅ Embedded and stored 31 chunks in Chroma
Collection: osds_chunks
Persist directory: ./chroma_db
